# 05 — Auto Curation

Filters sorted units by quality-metric thresholds and produces the outputs
consumed by `BurstDetectionTask`.

**Outputs written per well:**

| File | Contents |
|---|---|
| `quality_metrics.parquet` | Merged quality + template metrics with a `curated` boolean column |
| `rejection_log.parquet` | One row per rejected unit listing rejection reasons |
| `curated_spike_times.npy` | `dict[unit_id → np.ndarray]` of spike times in seconds |

**Default thresholds** (all configurable in `pipeline_config.json`):

| Metric | Default threshold |
|---|---|
| `presence_ratio` | ≥ 0.75 |
| `rp_contamination` | ≤ 0.15 |
| `firing_rate` | ≥ 0.05 Hz |
| `amplitude_median` | ≤ −20 µV |

Set `tasks.auto_curation.enabled = false` in `pipeline_config.json` to skip
filtering — all units will be marked `curated = true`.

In [ ]:
import sys
import traceback
import logging
from pathlib import Path

import pandas as pd

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager, WorkItem
from pipeline_manager.task_record import TaskStatus
from pipeline_tasks import AutoCurationTask
from config_manager import ConfigManager

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

## Import curation task

In [ ]:
print(f"Task: {AutoCurationTask.task_name!r}  deps={AutoCurationTask.dependencies}")
AutoCurationTask.default_params()

## Config

On the **first run** this cell writes a template and stops — edit the file then re-run.

Key parameters:
- `tasks.auto_curation.analyzer_output_root` — must match `tasks.analyzer.output_root`
- `tasks.auto_curation.enabled` — set `false` to bypass filtering
- Threshold keys: `presence_ratio_min`, `rp_contamination_max`, `firing_rate_min`,
  `amplitude_median_max`

In [ ]:
CONFIG_FILE = Path("../pipeline_config.json")

cm = ConfigManager()
cm.register_task(AutoCurationTask)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it, then re-run this cell."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded from: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print(f"  task params:   {cm.get_task_params(AutoCurationTask.task_name)}")

## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_by("scan_type", "==", "Network")

print(f"Found {len(recordings)} Network recording(s)")

summary_rows = [
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "n_wells":       len(r.wells),
    }
    for r in recordings
]
pd.DataFrame(summary_rows)

## 2. Register task and add wells

In [ ]:
TASK_NAME = AutoCurationTask.task_name

pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

try:
    pipeline_mgr.register_task(AutoCurationTask)
    print(f"Registered task: {TASK_NAME!r}")
except ValueError as e:
    print(f"Task already registered ({e}) — continuing")

n_added = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Skipped: {n_skipped}")

## 3. Pipeline status overview

In [ ]:
def _status_table(mgr: PipelineManager, task_name: str) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        rec_name, well_id = entry.well_id.split("/", 1)
        task = entry.tasks.get(task_name)
        rows.append({
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
            "status":        task.status if task else "\u2014",
            "config_fresh":  mgr.is_task_complete(WorkItem(entry.recording_key, entry.well_id, task_name))
                             if task and task.status == TaskStatus.COMPLETE else "\u2014",
            "output_path":   str(task.output_path) if task and task.output_path else "",
            "error":         (task.error or "")[:120] if task else "",
        })
    return pd.DataFrame(rows)


df_status = _status_table(pipeline_mgr, TASK_NAME)
print(df_status["status"].value_counts().to_string())
df_status

## 4. Run curation

Each well typically completes in under a minute (no waveform I/O — only
reads the pre-computed extension tables).

To retry failed wells: set `RETRY_FAILED = True`.  
To tune thresholds and re-run: edit `pipeline_config.json` and re-run —
the config-snapshot check will detect the change and mark all wells stale.

In [ ]:
RETRY_FAILED = False

task   = AutoCurationTask()
params = cm.get_task_params(TASK_NAME)

_rec_lookup = {r.cache_key: r for r in recordings}

while True:
    work_items = pipeline_mgr.get_next_task(n=1, retry_failed=RETRY_FAILED)
    if not work_items:
        break

    item      = work_items[0]
    rec_entry = _rec_lookup[item.recording_key]
    rec_name, well_id = item.well_id.split("/", 1)

    print(f"\nProcessing: {item.recording_key} / {rec_name} / {well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)

    try:
        output_path = task.run(
            item.recording_key,
            item.well_id,
            dataset_mgr.get_path(rec_entry),
            params,
        )
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output_path)
        print(f"  \u2713  saved \u2192 {output_path}")
    except Exception as exc:
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  \u2717  failed: {exc}")

print("\nDone \u2014 no more pending tasks.")

## 5. Final status report

In [ ]:
df_final = _status_table(pipeline_mgr, TASK_NAME)

print("=== Summary ===")
print(df_final["status"].value_counts().to_string())

stale = df_final[
    (df_final["status"] == TaskStatus.COMPLETE) & (df_final["config_fresh"] == False)
]
if not stale.empty:
    print(f"\n\u26a0\ufe0f  {len(stale)} well(s) completed with outdated config — "
          "edit thresholds in pipeline_config.json and pipeline_mgr.refresh(TASK_NAME) to re-run.")

failed = df_final[df_final["status"] == TaskStatus.FAILED]
if not failed.empty:
    print("\n=== Failed entries ===")
    for _, row in failed.iterrows():
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    error: {row['error']}")

df_final

## 6. Inspect curation results

Load `quality_metrics.parquet` for one completed well to review the
pass/reject distribution and per-metric statistics.

In [ ]:
# Pick the first completed well for inspection
completed = df_final[df_final["status"] == TaskStatus.COMPLETE]

if completed.empty:
    print("No completed wells yet — run the loop above first.")
else:
    row = completed.iloc[0]
    qm_path = Path(row["output_path"]) / "quality_metrics.parquet"
    rl_path = Path(row["output_path"]) / "rejection_log.parquet"

    qm = pd.read_parquet(qm_path)
    rl = pd.read_parquet(rl_path)

    n_total    = len(qm)
    n_curated  = qm["curated"].sum()
    n_rejected = n_total - n_curated

    print(f"Well: {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
    print(f"  Total units:    {n_total}")
    print(f"  Passed (curated): {n_curated}")
    print(f"  Rejected:         {n_rejected}")

    if n_rejected > 0 and not rl.empty:
        reason_counts = (
            rl["reasons"]
            .str.split("; ")
            .explode()
            .value_counts()
        )
        print("\nRejection reasons:")
        print(reason_counts.to_string())

    print("\nMetric summary (curated units):")
    metric_cols = ["presence_ratio", "rp_contamination", "firing_rate", "amplitude_median"]
    display_cols = [c for c in metric_cols if c in qm.columns]
    print(qm.loc[qm["curated"], display_cols].describe().round(3).to_string())

### All-wells curation summary

In [ ]:
summary_rows = []

for _, row in df_final[df_final["status"] == TaskStatus.COMPLETE].iterrows():
    qm_path = Path(row["output_path"]) / "quality_metrics.parquet"
    if not qm_path.exists():
        continue
    qm = pd.read_parquet(qm_path)
    summary_rows.append({
        "recording_key": row["recording_key"],
        "rec_name":      row["rec_name"],
        "well_id":       row["well_id"],
        "n_total":       len(qm),
        "n_curated":     int(qm["curated"].sum()),
        "pct_kept":      round(100 * qm["curated"].mean(), 1),
        "median_fr":     round(qm.loc[qm["curated"], "firing_rate"].median(), 3)
                         if "firing_rate" in qm.columns else None,
        "median_amp":    round(qm.loc[qm["curated"], "amplitude_median"].median(), 1)
                         if "amplitude_median" in qm.columns else None,
    })

pd.DataFrame(summary_rows)